In [1]:
# Cell 1 - Setup
from dotenv import load_dotenv
from openai import OpenAI
import base64
import os

load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    default_headers={"HTTP-Referer": "http://localhost"}
)

# Free vision models on OpenRouter
model = "google/gemini-2.5-pro-exp-03-25:free"  # best free vision model
# or
# model = "meta-llama/llama-3.2-11b-vision-instruct:free"

In [2]:
# Check available free vision models
import requests
import os

response = requests.get(
    "https://openrouter.ai/api/v1/models",
    headers={"Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}"}
)

models = response.json()["data"]

# Filter free vision models
free_vision = [
    m["id"] for m in models
    if m.get("pricing", {}).get("prompt") == "0"
    and (
        "vision" in str(m.get("architecture", {}).get("modality", "")).lower()
        or "gemini" in m["id"].lower()
        or "llava" in m["id"].lower()
        or "vision" in m["id"].lower()
    )
    and m.get("pricing", {}).get("prompt") == "0"
]

print("Free vision models available:")
for m in free_vision:
    print(f"  {m}")

Free vision models available:


In [3]:
# Cell 2 - Helper functions
def add_user_message(messages, message):
    messages.append({"role": "user", "content": message})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0):
    all_messages = []
    if system:
        all_messages.append({"role": "system", "content": system})
    all_messages += messages

    response = client.chat.completions.create(
        model=model,
        max_tokens=4000,
        messages=all_messages,
        temperature=temperature,
    )
    return response

def text_from_message(response):
    return response.choices[0].message.content or ""

In [4]:
# Cell 3 - Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected canopies
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation, limited space

For each item (1-5), write one sentence summarizing findings.
Final response: numeric Fire Risk Rating (1-4) with brief justification.
"""

In [5]:
# Cell 4 - Load image from file and send to model
def load_image_as_base64(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def get_image_media_type(image_path):
    ext = image_path.lower().split(".")[-1]
    types = {"jpg": "image/jpeg", "jpeg": "image/jpeg",
             "png": "image/png", "gif": "image/gif", "webp": "image/webp"}
    return types.get(ext, "image/jpeg")

def analyze_image_file(image_path, prompt):
    image_data = load_image_as_base64(image_path)
    media_type = get_image_media_type(image_path)

    messages = [{
        "role": "user",
        "content": [
            {
                "type": "image_url",           # ← OpenRouter uses image_url
                "image_url": {
                    "url": f"data:{media_type};base64,{image_data}"
                }
            },
            {
                "type": "text",
                "text": prompt
            }
        ]
    }]

    response = client.chat.completions.create(
        model=model,
        max_tokens=4000,
        messages=messages,
    )
    return text_from_message(response)

In [ ]:
# Cell 5 - Analyze all images in the images folder
images_folder = "images"  # ← your folder name

# Get all image files from the folder
image_files = [f for f in os.listdir(images_folder) 
               if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp", ".gif"))]

print(f"Found {len(image_files)} images: {image_files}")

# Analyze each image
for image_file in image_files:
    image_path = os.path.join(images_folder, image_file)
    print(f"\n{'='*50}")
    print(f"Analyzing: {image_file}")
    print('='*50)
    
    result = analyze_image_file(image_path, prompt)
    print(result)

In [ ]:
# Cell 6 - Analyze image from URL (no download needed)
def analyze_image_url(image_url, prompt):
    messages = [{
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {"url": image_url}   # ← direct URL works too
            },
            {
                "type": "text",
                "text": prompt
            }
        ]
    }]

    response = client.chat.completions.create(
        model=model,
        max_tokens=4000,
        messages=messages,
    )
    return text_from_message(response)

# Test with a sample satellite image URL
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/1/1a/24701-nature-natural-beauty.jpg/1280px-24701-nature-natural-beauty.jpg"
result = analyze_image_url(url, prompt)
print(result)